<a href="https://colab.research.google.com/github/monishabhojkumar12-avi/Aerial-Object-classification-and-detection/blob/main/Aerial_object.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/classification_dataset.zip"
extract_path = "/content/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped successfully ✅")

Unzipped successfully ✅


In [ ]:
import os
os.listdir("/content/classification_dataset")

['valid', 'test', 'train']

In [ ]:
import os

file_path = "classification_dataset/train/bird.cache"

if os.path.exists(file_path):
    os.remove(file_path)
    print("✅ Cache file deleted")
else:
    print("⚠️ File not found")

✅ Cache file deleted


In [ ]:
import os

file_path = "classification_dataset/valid/bird.cache"

if os.path.exists(file_path):
    os.remove(file_path)
    print("✅ Cache file deleted")
else:
    print("⚠️ File not found")

✅ Cache file deleted


In [ ]:
os.listdir("/content/classification_dataset/train")

['drone', 'bird']

In [ ]:
def count_images(path):
    for cls in os.listdir(path):
        class_path = os.path.join(path, cls)
        if os.path.isdir(class_path):
            print(f"{cls}: {len(os.listdir(class_path))}")

print("Train Data:")
count_images("/content/classification_dataset/train")

print("\nValidation Data:")
count_images("/content/classification_dataset/valid")

print("\nTest Data:")
count_images("/content/classification_dataset/test")

Train Data:
drone: 1248
bird: 1414

Validation Data:
drone: 225
bird: 217

Test Data:
drone: 94
bird: 121


In [ ]:
import os

print("Train:", os.listdir("/content/classification_dataset/train"))
print("Valid:", os.listdir("/content/classification_dataset/valid"))
print("Test:", os.listdir("/content/classification_dataset/test"))

Train: ['drone', 'bird']
Valid: ['drone', 'bird']
Test: ['drone', 'bird']


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models


In [ ]:
from tensorflow.keras.preprocessing import image_dataset_from_directory

train_data = image_dataset_from_directory(
    "/content/classification_dataset/train",
    image_size=(224,224),
    batch_size=32
)

val_data = image_dataset_from_directory(
    "/content/classification_dataset/valid",
    image_size=(224,224),
    batch_size=32
)

Found 2662 files belonging to 2 classes.
Found 442 files belonging to 2 classes.


In [ ]:
from tensorflow.keras import models, layers

model = models.Sequential([

    layers.Input(shape=(224,224,3)),

    # Normalize
    layers.Rescaling(1./255),

    # -------- BLOCK 1 --------
    layers.Conv2D(32, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # -------- BLOCK 2 --------
    layers.Conv2D(64, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # -------- BLOCK 3 --------
    layers.Conv2D(128, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # -------- BLOCK 4 (NEW - important) --------
    layers.Conv2D(256, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # -------- HEAD --------
    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    # Output
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 28, 28, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             

 Total params: 1,275,937 (4.87 MB)

 Trainable params: 1,273,505 (4.86 MB)

 Non-trainable params: 2,432 (9.50 KB)

In [ ]:
from tensorflow.keras import models, layers

model = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Rescaling(1./255),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),


    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,785 (432.75 KB)

 Trainable params: 110,337 (431.00 KB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
history = model.fit(train_data, validation_data=val_data, epochs=15)

Epoch 1/15
84/84 ━━━━━━━━━━━━━━━━━━━━ 1149s 14s/step - accuracy: 0.6912 - loss: 0.7403 - val_accuracy: 0.5090 - val_loss: 1.1228
Epoch 2/15
84/84 ━━━━━━━━━━━━━━━━━━━━ 1198s 14s/step - accuracy: 0.7014 - loss: 0.6745 - val_accuracy: 0.4910 - val_loss: 0.9674
Epoch 3/15
84/84 ━━━━━━━━━━━━━━━━━━━━ 1218s 14s/step - accuracy: 0.7194 - loss: 0.5703 - val_accuracy: 0.4910 - val_loss: 1.0085
Epoch 4/15
84/84 ━━━━━━━━━━━━━━━━━━━━ 1205s 14s/step - accuracy: 0.7494 - loss: 0.5292 - val_accuracy: 0.6222 - val_loss: 1.0291
Epoch 5/15
84/84 ━━━━━━━━━━━━━━━━━━━━ 1150s 14s/step - accuracy: 0.7276 - loss: 0.5559 - val_accuracy: 0.4932 - val_loss: 1.2586
Epoch 6/15
84/84 ━━━━━━━━━━━━━━━━━━━━ 1193s 14s/step - accuracy: 0.7806 - loss: 0.4865 - val_accuracy: 0.4977 - val_loss: 2.0208
Epoch 7/15
84/84 ━━━━━━━━━━━━━━━━━━━━ 1199s 14s/step - accuracy: 0.7832 - loss: 0.4465 - val_accuracy: 0.5181 - val_loss: 1.3328
Epoch 8/15
84/84 ━━━━━━━━━━━━━━━━━━━━ 1189s 14s/step - accuracy: 0.7517 - loss: 0.5237 - val_accu

In [ ]:
model.save("/content/drive/MyDrive/aerial_dataset/cnn_model.h5")

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("/content/drive/MyDrive/aerial_dataset/cnn_model.h5")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


In [ ]:
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]
)

val_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [ ]:
train_data = train_gen.flow_from_directory(
    "/content/classification_dataset/train",
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

val_data = val_gen.flow_from_directory(
    "/content/classification_dataset/valid",
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

Found 2662 images belonging to 2 classes.
Found 442 images belonging to 2 classes.


In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)


base_model.trainable = False


x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(1, activation='sigmoid')(x)

cnn_model = models.Model(inputs=base_model.input, outputs=output)

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,785 (432.75 KB)

 Trainable params: 110,337 (431.00 KB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ModelCheckpoint("best_mobilenet.h5", save_best_only=True)
]

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.7977 - loss: 0.4392

84/84 ━━━━━━━━━━━━━━━━━━━━ 474s 6s/step - accuracy: 0.7975 - loss: 0.4421 - val_accuracy: 0.5090 - val_loss: 7.8949
Epoch 2/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.7924 - loss: 0.4445

84/84 ━━━━━━━━━━━━━━━━━━━━ 469s 6s/step - accuracy: 0.7919 - loss: 0.4352 - val_accuracy: 0.7036 - val_loss: 1.0875
Epoch 3/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 503s 6s/step - accuracy: 0.7968 - loss: 0.4297 - val_accuracy: 0.5090 - val_loss: 18.2953
Epoch 4/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.8123 - loss: 0.4170

84/84 ━━━━━━━━━━━━━━━━━━━━ 500s 6s/step - accuracy: 0.8028 - loss: 0.4305 - val_accuracy: 0.5588 - val_loss: 1.0284
Epoch 5/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 475s 6s/step - accuracy: 0.7990 - loss: 0.4281 - val_accuracy: 0.4910 - val_loss: 29.2696
Epoch 6/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.8207 - loss: 0.4022

84/84 ━━━━━━━━━━━━━━━━━━━━ 467s 6s/step - accuracy: 0.8125 - loss: 0.4093 - val_accuracy: 0.6335 - val_loss: 0.7816
Epoch 7/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 471s 6s/step - accuracy: 0.8144 - loss: 0.3999 - val_accuracy: 0.5362 - val_loss: 2.7747
Epoch 8/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 499s 6s/step - accuracy: 0.8069 - loss: 0.4028 - val_accuracy: 0.4253 - val_loss: 9.4608
Epoch 9/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 473s 6s/step - accuracy: 0.8174 - loss: 0.4000 - val_accuracy: 0.5090 - val_loss: 30.2983


In [ ]:

for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # low learning rate
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 478s 6s/step - accuracy: 0.8197 - loss: 0.4029 - val_accuracy: 0.6335 - val_loss: 1.0294
Epoch 2/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 504s 6s/step - accuracy: 0.8193 - loss: 0.3942 - val_accuracy: 0.7511 - val_loss: 0.5604
Epoch 3/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 469s 6s/step - accuracy: 0.8287 - loss: 0.3923 - val_accuracy: 0.7986 - val_loss: 0.4093
Epoch 4/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 468s 6s/step - accuracy: 0.8246 - loss: 0.3874 - val_accuracy: 0.8235 - val_loss: 0.3926
Epoch 5/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 478s 6s/step - accuracy: 0.8306 - loss: 0.3871 - val_accuracy: 0.8190 - val_loss: 0.3965


In [ ]:
model.save("/content/drive/MyDrive/aerial_dataset/mobilenet_model.h5")

In [ ]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

--2026-04-07 12:38:48--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64 [following]
--2026-04-07 12:38:48--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/731ab2f8-6b77-4adb-a7b3-1104525e9d72?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-07T13%3A15%3A22Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-07T1

In [ ]:
%%writefile model.py
import streamlit as st
import numpy as np
from PIL import Image
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

@st.cache_resource
def load_models():
    cnn = load_model("/content/drive/MyDrive/aerial_dataset/cnn_model.h5")
    mobilenet = load_model("/content/drive/MyDrive/aerial_dataset/mobilenet_model.h5")
    return cnn, mobilenet

cnn_model, mobilenet_model = load_models()


st.set_page_config(page_title="Aerial Classifier", layout="centered")

st.title("🛰️ Aerial Object Classifier")
st.write("Upload an image to classify Bird 🐦 or Drone 🛸")


model_choice = st.selectbox(
    "Choose Model",
    ["CNN Model", "MobileNet Model"]
)

uploaded_file = st.file_uploader("Upload Image", type=["jpg", "png", "jpeg"])


if uploaded_file is not None:
    image = Image.open(uploaded_file)
    st.image(image, caption="Uploaded Image", use_column_width=True)


    if st.button("🔍 Predict"):

        img = image.resize((224,224))
        img = np.array(img)


        if model_choice == "CNN Model":
            img = img / 255.0
            model = cnn_model
        else:
            img = preprocess_input(img)
            model = mobilenet_model

        img = np.expand_dims(img, axis=0)


        with st.spinner("Predicting..."):
            prediction = model.predict(img)[0][0]


        st.subheader("📌 Prediction Result")

        if prediction > 0.5:
            label = "🛸 Drone"
            confidence = prediction
        else:
            label = "🐦 Bird"
            confidence = 1 - prediction

        st.success(f"Prediction: {label}")
        st.write(f"Confidence: {confidence:.2f}")

Overwriting model.py


In [ ]:
!streamlit run /content/model.py &>/content/logs.txt &

In [ ]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://fundamentals-manufacturing-massive-pursue.trycloudflare.com
